Making of the Machine Learning Model.

Stage 1: Baseline Logistic Regression

initially we will run a logistic regression that will serve as a baseline for our project

In [123]:
import pandas as pd
import numpy as np 


In [124]:
df=pd.read_csv("master_dataset_fixed.csv")
df.head()

,raceId,year,round,circuitId,race_name,circuit_name,location,country,driverStandingsId,driverId,...,final_position,win_flag,driver_momentum,constructor_momentum,driver_circuit_specialist_score,team_avg_quali,driver_quali_strength,constructor_avg_lap,rel_race_pace,driver_performance_index
0,989,2018,1,1,Australian Grand Prix,Albert Park Grand Prix Circuit,Melbourne,Australia,68610,1,...,2,0.0,0.000,0.0,7.0,10.317726,9.317726,97.086366,4.356728,19.445928
1,990,2018,2,3,Bahrain Grand Prix,Bahrain International Circuit,Sakhir,Bahrain,68690,1,...,3,0.0,1.000,0.0,7.0,10.500000,6.500000,89.384104,-7.606282,21.598077
2,991,2018,3,17,Chinese Grand Prix,Shanghai International Circuit,Shanghai,China,68670,1,...,4,0.0,-0.750,1.0,7.0,10.317726,6.317726,97.086366,-5.652295,22.247732
3,992,2018,4,73,Azerbaijan Grand Prix,Baku City Circuit,Baku,Azerbaijan,68710,1,...,1,1.0,-0.250,-1.0,7.0,10.317726,8.317726,97.086366,-24.958556,25.408984
4,993,2018,5,4,Spanish Grand Prix,Circuit de Barcelona-Catalunya,Montmeló,Spain,68730,1,...,1,1.0,-0.125,1.0,7.0,10.317726,9.317726,97.086366,10.268608,18.063552


In [125]:
leakage_cols = [
    'final_position',                 # actual race result — must NEVER be in X
    'points',                         # championship points after the race
    'wins',                           # wins after this race
    'constructor_final_position',     # leaks constructor outcome
    'points_constructor',             # leaks season result
    'wins_constructor',               # leaks season standings
    'driverStandingsId',              # position in standings after race
    'constructorStandingsId',         # same leakage
]


In [126]:
df['win_flag'] = (df['final_position'] == 1).astype(int)
non_numeric_cols = df.select_dtypes(include=['object']).columns.tolist()
X = df.drop(columns=non_numeric_cols + leakage_cols + ['win_flag'], errors='ignore')
X = X.apply(pd.to_numeric, errors='coerce')
y = df['win_flag']

df.shape, X.shape

((2578, 55), (2578, 37))

In [127]:
feature_columns = X.columns.tolist()


In [128]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)




In [129]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy='mean')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp  = imputer.transform(X_test)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled  = scaler.transform(X_test_imp)



In [130]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
logreg = LogisticRegression(max_iter=3000, class_weight='balanced')
logreg.fit(X_train_scaled, y_train)




,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,'balanced'
,random_state,None
,solver,'lbfgs'
,max_iter,3000
,multi_class,'deprecated'


predicting win probabilities and podium for a specified race with raceId being the id of the race we want predictions for. model is the logistic regression

In [131]:
def predict_race_results(race_id, df, model, imputer, scaler, feature_columns):


    race_df = df[df['raceId'] == race_id].copy()
    if race_df.empty:
        print("No race with this raceId found.")
        return None
    X_race = race_df[feature_columns].copy()
    X_race = X_race.apply(pd.to_numeric, errors='coerce')
    X_race_imp = imputer.transform(X_race)
    X_race_scaled = scaler.transform(X_race_imp)
    race_df['predicted_win_probability'] = model.predict_proba(X_race_scaled)[:, 1]
    race_df['predicted_rank'] = race_df['predicted_win_probability'] \
        .rank(ascending=False, method='first')
    race_df = race_df.sort_values('predicted_win_probability', ascending=False)
     # Actual winner (from real race results)
    actual_winner_row = df[(df['raceId'] == race_id) & (df['final_position'] == 1)]
    if actual_winner_row.empty:
        actual_winner = None
    else:
        actual_winner = actual_winner_row.iloc[0]['driver_name']
    predicted_winner = race_df.iloc[0]['driver_name']
    winner_correct = "Yes" if actual_winner == predicted_winner else "No"
    podium = race_df.head(3)[['race_name', 'year', 'driver_name',  'predicted_win_probability']]
    return race_df, podium, actual_winner, predicted_winner, winner_correct


now in order to call this we will run the following code block. it will display the podium of the race in reference to the raceId specified

In [132]:
race_id = 1111

full_results, podium, actual, predicted, correct = predict_race_results(
    race_id=race_id,
    df=df,
    model=logreg,
    imputer=imputer,
    scaler=scaler,
    feature_columns=feature_columns
)

print("Predicted Podium:")
display(podium)

print(f"Actual Winner: {actual}")
print(f"Predicted Winner: {predicted}")
print(f"Correct Prediction? {correct}")




Predicted Podium:


,race_name,year,driver_name,predicted_win_probability
1162,Dutch Grand Prix,2023,Max Verstappen,0.983248
641,Dutch Grand Prix,2023,Sergio Pérez,0.771348
205,Dutch Grand Prix,2023,Fernando Alonso,0.680467


Actual Winner: Max Verstappen
Predicted Winner: Max Verstappen
Correct Prediction? Yes


In [136]:
from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
y_pred = logreg.predict(X_test_scaled)
y_proba = logreg.predict_proba(X_test_scaled)[:, 1]
print("Accuracy", accuracy_score(y_test, y_pred))
print("ROC AUC", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

Accuracy 0.8565891472868217
ROC AUC 0.9173525377229081
              precision    recall  f1-score   support

           0       0.98      0.86      0.92       486
           1       0.26      0.77      0.38        30

    accuracy                           0.86       516
   macro avg       0.62      0.81      0.65       516
weighted avg       0.94      0.86      0.89       516



In [ ]:
list(X.columns)


['raceId',
 'year',
 'round',
 'circuitId',
 'driverId',
 'year_y',
 'constructorId',
 'qualifying_position',
 'q1',
 'q2',
 'q3',
 'pit_stop_count',
 'pit_duration',
 'avg_lap_time',
 'total_laps',
 'pos_ratio',
 'driver_form',
 'driver_circuit_performance',
 'driver_circuit_performance_5',
 'avg_time_per_stop',
 'constructor_avg_pit',
 'constructor_circuit_avg_finish',
 'driver_circuit_experience',
 'driver_qualifying_form',
 'team_race_avg_finish',
 'rel_teammate_performance',
 'driver_teammate_diff_form',
 'driver_teammate_diff_historic',
 'driver_consistency_std']